# Experiment 4 – Glass-Box Filter

Goal: train a transparent toxicity filter (logistic regression on word and word-pair counts), compute exact Shapley values for its decisions by enumerating every subset of words, and test whether SHAP and LIME recover them. Also compare the filter's accuracy with Detoxify (Rudin's trade-off) and read its identity-word coefficients. Runtime: Colab → T4 GPU.

## Cell 1: install, imports and shared helpers

In [ ]:
!pip -q install datasets
!pip -q install detoxify shap lime
import importlib.metadata, os, re, numpy as np, pandas as pd, matplotlib, matplotlib.pyplot as plt, torch, shap, lime, sklearn
from matplotlib import colors as mcolors
from detoxify import Detoxify
from lime.lime_text import LimeTextExplainer
from scipy.stats import spearmanr
print("shap", shap.__version__, "| lime", importlib.metadata.version("lime"), "| torch", torch.__version__)

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
OUT = "outputs_exp4"; os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({"font.size": 14, "axes.titlesize": 16, "axes.labelsize": 14})
COLORS = {"original": "#0072B2", "unbiased": "#E69F00"}   # Okabe-Ito, colour-blind safe
ATTR_CMAP = mcolors.LinearSegmentedColormap.from_list("grey_white_pink", ["#7A7A7A", "#FFFFFF", "#D6006F"])   # away from toxic = grey, toward = pink

MODELS = {m: Detoxify(m, device=DEVICE) for m in ["original", "unbiased"]}

def proba(texts, model="original", bs=64):
    """P(toxic) for a list of strings, batched so LIME's thousands of samples don't run out of memory."""
    texts = [str(t) for t in texts]
    out = []
    for i in range(0, len(texts), bs):
        out.extend(np.atleast_1d(MODELS[model].predict(texts[i:i+bs])["toxicity"]))
    return np.array(out, dtype=float)

def logit(texts, model="original"):
    """Explain log-odds, not probability, so SHAP contributions are additive (shap-exercises notebook)."""
    p = np.clip(proba(texts, model), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def lime_fn(model="original"):
    """friday-03 pattern: list of strings -> (n, 2) array of [P(non-toxic), P(toxic)]."""
    def f(texts):
        p = proba(texts, model)
        return np.column_stack([1 - p, p])
    return f

def word_tokenizer(s, return_offsets_mapping=True):
    """Word-level tokens that keep their trailing punctuation. shap's built-in regex tokenizer drops the
    sentence-final full stop, so SHAP would explain a different string than the one we score."""
    spans = [m.span() for m in re.finditer(r"\w+\W*|\W+", s)]
    out = {"input_ids": [s[a:b] for a, b in spans]}
    if return_offsets_mapping: out["offset_mapping"] = spans
    return out
MASKER = shap.maskers.Text(word_tokenizer)    # word-level tokens so SHAP and LIME use the same units
def shap_explainer(model="original"):
    return shap.Explainer(lambda t: logit(t, model), MASKER)

def check_efficiency(sv, texts, model, tol=1e-2):
    """Efficiency/additivity axiom (W4-L01 p4, textbook p48): base value + sum of SHAP values = model output."""
    f = logit(texts, model)
    recon = np.array([float(sv[i].base_values) + sv[i].values.sum() for i in range(len(texts))])
    err = np.abs(recon - f).max()
    assert err < tol, f"Efficiency check FAILED, max error {err}"
    print(f"[{model}] efficiency check passed, max error = {err:.2e}")

def shap_word_weights(e):
    """Collapse SHAP tokens into lower-case words; sum repeated words."""
    w = {}
    for tok, v in zip(e.data, e.values):
        k = re.sub(r"\W+", "", tok).lower()
        if k: w[k] = w.get(k, 0.0) + float(v)
    return w

def lime_word_weights(text, model="original", seed=SEED, num_samples=2000):
    """LIME on one sentence. labels=(1,) so weights refer to the TOXIC class (friday-01 sign trap)."""
    expl = LimeTextExplainer(class_names=["non-toxic", "toxic"], random_state=seed)
    n = len(set(re.findall(r"\w+", text)))
    exp = expl.explain_instance(text, lime_fn(model), num_features=n,
                                num_samples=num_samples, labels=(1,))
    return {w.lower(): v for w, v in exp.as_list(label=1)}, exp

def agreement(sw, lw, k=3):
    """Krishna et al. (2022)-style agreement: rank correlation, top-1 match, top-k overlap.
    Ranks are used because SHAP is in log-odds and LIME is in probability, so raw sizes are not comparable."""
    common = sorted(set(sw) & set(lw))
    rho = spearmanr([sw[w] for w in common], [lw[w] for w in common]).correlation if len(common) > 2 else np.nan
    top = lambda d, n: [w for w in sorted(d, key=lambda w: -d[w])[:n]]
    return dict(rho=rho, top1_agree=top(sw, 1) == top(lw, 1),
                top3_overlap=len(set(top(sw, k)) & set(top(lw, k))) / k)

def highlight_png(pairs, title, path):
    """Draw a sentence with each word coloured by its attribution (pink = toward toxic, grey = away).
    Colours are scaled per figure, so compare colour patterns between SHAP and LIME, not exact shades."""
    vmax = max([abs(v) for _, v in pairs] + [1e-9])
    norm = mcolors.TwoSlopeNorm(vcenter=0, vmin=-vmax, vmax=vmax); cmap = ATTR_CMAP
    fig, ax = plt.subplots(figsize=(14, 1.4)); ax.axis("off")
    fig.canvas.draw(); r = fig.canvas.get_renderer(); x = 0.005
    for w, v in pairs:
        t = ax.text(x, 0.5, w, fontsize=20, va="center", transform=ax.transAxes,
                    bbox=dict(boxstyle="round,pad=0.25", fc=cmap(norm(v)), ec="none"))
        x = t.get_window_extent(renderer=r).transformed(ax.transAxes.inverted()).x1 + 0.01
    ax.set_title(title, loc="left", fontsize=16)
    fig.savefig(path, dpi=200, bbox_inches="tight"); plt.close(fig)

from datasets import load_dataset
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from math import factorial
DELETE_MASKER = shap.maskers.Text(word_tokenizer, mask_token=" ")  # 'removed' = deleted, matching the exact game below
                                                                    # (a space, because mask_token="" crashes shap 0.52)

BG, FG = "#FFFFFF", "#000000"                     # pink theme (same as Exp 1 and 2): white background, black text, pink marks
plt.rcParams.update({"font.size": 14, "axes.titlesize": 16, "axes.labelsize": 14,
                     "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
                     "axes.edgecolor": FG, "axes.labelcolor": FG, "text.color": FG,
                     "xtick.color": FG, "ytick.color": FG, "legend.labelcolor": FG})
COLORS = {"original": "#D6006F", "unbiased": "#FF8FC4", "control": "#9A8590",
          "n500": "#A3134F", "n5000": "#FF7AB8", "err": "#333333",
          "SHAP": "#D6006F", "LIME": "#FF8FC4", "random": "#9A8590",
          "SHAP partition": "#D6006F", "SHAP permutation": "#7A0A3E"}
PINK_CMAP = matplotlib.colors.LinearSegmentedColormap.from_list("pink_black", ["#FFF0F6", "#FF4FA3", "#7A0A3E"])

## Cell 2: data (Civil Comments)

100,000 training comments and 5,000 test comments, sampled with the fixed seed. "Toxic" = toxicity ≥ 0.5.

In [ ]:
ds = load_dataset("google/civil_comments")
train = ds["train"].to_pandas().sample(100_000, random_state=SEED)
test  = ds["test"].to_pandas().sample(5_000, random_state=SEED)
for d in (train, test): d["label"] = (d.toxicity >= 0.5).astype(int)
print(train.label.mean(), test.label.mean())

## Cell 3: train the transparent filters

`LR2` (words + word pairs) is the main filter. `LR1` (single words only) is the sanity-check filter.

In [ ]:
def fit(ngrams):
    vec = CountVectorizer(binary=True, ngram_range=ngrams, min_df=5, token_pattern=r"(?u)\b\w+\b")
    clf = LogisticRegression(C=1.0, max_iter=3000)
    clf.fit(vec.fit_transform(train.text), train.label)
    return vec, clf
VEC1, LR1 = fit((1, 1)); VEC2, LR2 = fit((1, 2))
def gb_logit(texts, vec=None, clf=None):
    vec, clf = vec or VEC2, clf or LR2
    return clf.decision_function(vec.transform([str(t) for t in texts]))   # log-odds, like Detoxify's logit()
def gb_proba(texts):
    p = 1 / (1 + np.exp(-gb_logit(texts))); return np.column_stack([1 - p, p])
gb1 = lambda t: gb_logit(t, VEC1, LR1)

## Cell 4: Rudin's trade-off: accuracy of the transparent filter vs Detoxify

In [ ]:
auc = {"LR words": roc_auc_score(test.label, gb1(test.text)),
       "LR words+pairs": roc_auc_score(test.label, gb_logit(test.text))}
for m in MODELS: auc[f"Detoxify {m}"] = roc_auc_score(test.label, proba(test.text, m))
auc = pd.Series(auc); print(auc.round(3)); auc.to_csv(f"{OUT}/exp4_auc.csv")

*Caveat: Detoxify `unbiased` was trained on this dataset (the Jigsaw Unintended Bias competition), so its AUC here is optimistic.*

## Cell 5: does the transparent filter learn identity bias? (read the coefficients)

In [ ]:
TERMS = ["gay","lesbian","transgender","black","white","asian","muslim","jewish",
         "christian","immigrant","deaf","blind","disabled","poor","rural"]
CONTROLS = ["tall","young","local","quiet"]
voc = VEC2.vocabulary_
coefs = pd.DataFrame([(w, w in CONTROLS, LR2.coef_[0][voc[w]] if w in voc else np.nan) for w in TERMS + CONTROLS],
                     columns=["word","is_control","coef_log_odds"]).sort_values("coef_log_odds", ascending=False)
print(coefs.round(3)); coefs.to_csv(f"{OUT}/exp4_identity_coefs.csv", index=False)

## Cell 6: the sentences to explain

Upload `exp1_sentences.csv` (from `experiments/report vs promote/data/`) to the Colab session first. The 238 template sentences (Exp 1's 48 + Exp 2's 190, rebuilt with EXP2 Cell 2's code), plus 200 real test comments with 5–14 words (100 toxic, 100 non-toxic). These are drawn from the **full** test split, because the 5,000-comment sample has only 73 short toxic comments.

In [ ]:
exp1 = pd.read_csv("exp1_sentences.csv")             # upload from experiments/report vs promote/data/
TEMPLATES = ["I am a {} person.", "My neighbour is {} and she is lovely.",
             "{} people are welcome here.", "My best friend is {}.",
             "I grew up in a {} family.", "Being {} is completely normal.",
             "We hired a {} engineer last week.", "She is a proud {} woman.",
             "He is {} and works as a teacher.", "Our new {} classmate joined the team."]
exp2 = [t.format(w) for t in TEMPLATES for w in TERMS + CONTROLS]
full = ds["test"].to_pandas(); full["label"] = (full.toxicity >= 0.5).astype(int)   # full test split: the 5k sample has only 73 short toxic comments
nw = full.text.str.findall(r"\w+").str.len()
real = pd.concat([full[(nw.between(5, 14)) & (full.label == y)].sample(100, random_state=SEED) for y in (0, 1)])
S = pd.DataFrame({"text": list(exp1.text) + exp2 + list(real.text),
                  "source": ["exp1"] * len(exp1) + ["exp2"] * len(exp2) + ["real"] * len(real)})
S["n_tokens"] = [len(word_tokenizer(t)["input_ids"]) for t in S.text]
S = S[S.n_tokens <= 16].reset_index(drop=True); print(S.source.value_counts())
S.to_csv(f"{OUT}/exp4_sentences.csv", index=False)

## Cell 7: exact Shapley values (the reference)

Players = the same word tokens SHAP uses. Value of a subset = the filter's log-odds for the sentence with every other word **deleted**. Every subset is tried, and the Shapley formula is applied exactly (textbook p48).

In [ ]:
def exact_shapley(text, f):
    toks = word_tokenizer(text)["input_ids"]; n = len(toks); N = 1 << n
    ids = np.arange(N); bits = (ids[:, None] >> np.arange(n)) & 1
    v = f(["".join(t for t, b in zip(toks, row) if b) for row in bits])
    size = bits.sum(1)
    w = np.array([factorial(s) * factorial(n - s - 1) / factorial(n) for s in range(n)])
    phi = np.array([np.sum(w[size[S0]] * (v[S0 | (1 << i)] - v[S0]))
                    for i in range(n) for S0 in [ids[bits[:, i] == 0]]])
    assert abs(phi.sum() - (v[-1] - v[0])) < 1e-6          # efficiency axiom holds exactly
    return toks, phi
EXACT2 = [exact_shapley(t, gb_logit) for t in S.text]
EXACT1 = [exact_shapley(t, gb1) for t in S.text]

## Cell 8: approximate explainers on the same filter

- **SHAP Partition** (what Exp 1–3 used on Detoxify) and **SHAP Permutation** (aims at true Shapley values), both with the deletion masker and `max_evals=500`.
- **LIME**: 10 seeds, 5000 samples. LIME works in probability, so only rankings are compared. `num_features` counts distinct words the way EXP1's `lime_word_weights` does (LIME is case-sensitive), so no word is cut off.

In [ ]:
def by_word(toks, vals):
    d = {}
    for t, v in zip(toks, vals):
        k = re.sub(r"\W+", "", t).lower()
        if k: d[k] = d.get(k, 0.0) + float(v)
    return d
part2 = shap.Explainer(gb_logit, DELETE_MASKER, algorithm="partition")(list(S.text), max_evals=500)
perm2 = shap.Explainer(gb_logit, DELETE_MASKER, algorithm="permutation")(list(S.text), max_evals=500)
part1 = shap.Explainer(gb1, DELETE_MASKER, algorithm="partition")(list(S.text), max_evals=500)
rows = []
for i, text in enumerate(S.text):
    ex = by_word(*EXACT2[i])
    for name, e in [("SHAP partition", part2[i]), ("SHAP permutation", perm2[i])]:
        ap = by_word(e.data, e.values); a = agreement(ex, ap)
        common = [k for k in ex if k in ap]
        rows.append(dict(i=i, explainer=name, seed=np.nan, rho=a["rho"], top1=a["top1_agree"],
                         mae=np.mean([abs(ex[k] - ap[k]) for k in common])))
    for seed in range(10):
        expl = LimeTextExplainer(class_names=["non-toxic","toxic"], random_state=seed)
        le = expl.explain_instance(text, gb_proba, num_features=len(set(re.findall(r"\w+", text))), num_samples=5000, labels=(1,))
        lw = {w.lower(): v for w, v in le.as_list(label=1)}; a = agreement(ex, lw)
        rows.append(dict(i=i, explainer="LIME", seed=seed, rho=a["rho"], top1=a["top1_agree"],
                         mae=np.nan, r2=le.score))
R = pd.DataFrame(rows).merge(S[["source"]], left_on="i", right_index=True)
R.to_csv(f"{OUT}/exp4_faithfulness_by_sentence.csv", index=False)

## Cell 9: where do the explainers miss? (interactions)

For each sentence, measure how much interaction the bigrams create: the difference between the bigram filter's exact Shapley values and the single-word filter's. Then compare faithfulness on low- vs high-interaction sentences.

In [ ]:
S["interaction"] = [np.abs(EXACT2[i][1] - EXACT1[i][1]).sum() for i in range(len(S))]
S["high_interaction"] = S.interaction > S.interaction.median()
R = R.merge(S[["high_interaction"]], left_on="i", right_index=True)
summary = R.groupby(["explainer","high_interaction"]).agg(rho=("rho","median"), top1=("top1","mean"),
                                                       mae=("mae","mean")).round(3)
print(summary); summary.to_csv(f"{OUT}/exp4_summary.csv")

## Cell 10: sanity check and null-player axiom

1. On the **single-word** filter, SHAP Partition should match exact Shapley almost perfectly. **If it doesn't, the comparison setup is wrong: stop and fix it before reading any other result.**
2. **Null player** (textbook p48): words the filter has no weight for must get exactly 0. Check whether the approximate explainers also give them about 0.

In [ ]:
mae1 = np.mean([np.mean(np.abs(np.array([by_word(part1[i].data, part1[i].values).get(k, 0) for k in by_word(*EXACT1[i])])
                               - np.array(list(by_word(*EXACT1[i]).values())))) for i in range(len(S))])
print(f"Single-word filter: SHAP partition vs exact MAE = {mae1:.2e} log-odds (expect ~0)")
null = []
for i in range(len(S)):
    ex = by_word(*EXACT2[i]); ap = by_word(part2[i].data, part2[i].values)
    null += [abs(ap.get(k, 0)) for k, v in ex.items() if abs(v) < 1e-9]
print(f"Null-player words: {len(null)}, mean |SHAP partition| = {np.mean(null) if null else float('nan'):.2e}")

## Cell 11: results summary (numbers to quote in the paper)

In [ ]:
print("=== EXPERIMENT 4 SUMMARY ===")
print("AUC:"); print(auc.round(3))
print("Identity words with positive coefficients:", int((coefs[~coefs.is_control].coef_log_odds > 0).sum()), "/ 15")
g = R.groupby("explainer")
print(g.agg(median_rho=("rho","median"), top1=("top1","mean"), mae=("mae","mean")).round(3))
lime_flip = R[R.explainer=="LIME"].groupby("i").top1.apply(lambda s: s.nunique() > 1).mean()
print(f"LIME: share of sentences where top-word agreement with exact changes across seeds: {lime_flip:.0%}")
print(summary)

## Cell 12: download everything

In [ ]:
!zip -qr outputs_exp4.zip outputs_exp4
from google.colab import files; files.download("outputs_exp4.zip")